Cell 1 — Inspect Dataset Structure

In [2]:
import os
import numpy as np
import pandas as pd
import ast
import pathlib


This code explores the dataset folder to list all top-level linguistic categories (like Adjectives, Verbs, etc.) and count how many there are. It then goes into each category, collects all sign subfolders, and computes how many unique signs exist in total. This helps verify that the dataset structure is correct and gives a quick overview of the class coverage before model training.

In [3]:

data_root = pathlib.Path("../data/archive/Dataset - MP - CSV")
categories = sorted([d.name for d in data_root.iterdir() if d.is_dir()])

print(f"Total categories : {len(categories)}")
print(f"Categories : {categories}")
all_signs = []

for cat in categories:
    signs = [d.name for d in (data_root / cat).iterdir() if d.is_dir()]
    all_signs.extend(signs)

print(f"Total unique signs: {len(set(all_signs))}")

Total categories : 16
Categories : ['Adjectives', 'Adverb', 'Colors', 'Conjunctions', 'Days', 'Determiner', 'Greetings', 'Interjection', 'Months', 'Nouns', 'Numbers', 'People', 'Places', 'Preposition', 'Vehicles', 'Verbs']
Total unique signs: 383


This step defines the constants and a helper function to load each CSV and convert it into a fixed-length sequence of landmark features.

In [4]:
SEQUENCE_LEN = 30
NUM_FEATURES = 132  # 33 landmarks x 4 values (x, y, z, visibility)

def csv_to_sequence(csv_path, seq_len=SEQUENCE_LEN):
    df = pd.read_csv(csv_path, header=None)
    frames = []
    for _, row in df.iterrows():
        frame_features = []
        for cell in row:
            try:
                coords = ast.literal_eval(str(cell))  # e.g. [0.45, 0.62, 0.01, 0.99]
                frame_features.extend(coords)
            except Exception:
                frame_features.extend([0.0, 0.0, 0.0, 0.0])
        frames.append(frame_features)
    # Make every clip exactly 30 frames (pad with zeros or trim)
    while len(frames) < seq_len:
        frames.append([0.0] * NUM_FEATURES)
    return np.array(frames[:seq_len])

This step converts each CSV clip into a fixed-length sequence, builds the feature array `X` and label list `y`, encodes labels into integers, and saves a `label_map.json` for decoding predictions later.

In [5]:
from sklearn.preprocessing import LabelEncoder
import json
from pathlib import Path

X, y = [], []
for category in categories:
    cat_path = data_root / category
    for sign_folder in cat_path.iterdir():
        if not sign_folder.is_dir():
            continue
        for csv_file in sign_folder.glob("*.csv"):
            seq = csv_to_sequence(csv_file)
            X.append(seq)
            y.append(sign_folder.name)  # e.g. "Ayubowan"
X = np.array(X)  # shape: (4236, 30, 132)
# Turn text labels into numbers (e.g. "Ayubowan" -> 0)
le = LabelEncoder()
y_enc = le.fit_transform(y)
# Save label map so the backend can decode predictions later
label_map = {int(i): cls for i, cls in enumerate(le.classes_)}
label_map_path = Path("../saved_models/label_map.json")
label_map_path.parent.mkdir(parents=True, exist_ok=True)
with label_map_path.open("w") as f:
    json.dump(label_map, f)
print(f"X shape : {X.shape}")
print(f"Classes : {len(le.classes_)}")

X shape : (4236, 30, 132)
Classes : 383


This step normalizes the sequences, one-hot encodes labels, splits the dataset into train/validation/test sets, and saves the resulting NumPy arrays to `ml/data` for later training.

In [7]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from pathlib import Path
# Scale all values to 0-1 range

X_min = X.min(axis=(0, 1), keepdims=True)
X_max = X.max(axis=(0, 1), keepdims=True)
X_norm = (X - X_min) / (X_max - X_min + 1e-8)

# Convert label numbers to one-hot format (e.g. 3 -> [0,0,0,1,0,...])
NUM_CLASSES = len(le.classes_)  # 383
y_cat = to_categorical(y_enc, num_classes=NUM_CLASSES)

# Split: 70% train | 15% val | 15% test
min_class_count = np.bincount(y_enc).min()
stratify_labels = y_enc if min_class_count >= 2 else None
if stratify_labels is None:
    print("Warning: too few samples per class; splitting without stratify.")

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_norm, y_cat, test_size=0.30, random_state=42, stratify=stratify_labels)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=None)

# Save everything to disk
out_dir = Path("../data")
out_dir.mkdir(parents=True, exist_ok=True)
np.save(out_dir / "X_train.npy", X_tr)
np.save(out_dir / "X_val.npy", X_val)
np.save(out_dir / "X_test.npy", X_test)
np.save(out_dir / "y_train.npy", y_tr)
np.save(out_dir / "y_val.npy", y_val)
np.save(out_dir / "y_test.npy", y_test)
print("Done! Files saved.")
print(f"Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Done! Files saved.
Train: (2965, 30, 132) | Val: (635, 30, 132) | Test: (636, 30, 132)
